S1 Question: Explain what Computer Vision is and how it differs from a simple rules-based image
filter. Why is a CV-based approach more suitable for categorising diverse food images than
writing fixed conditions based on colour thresholds or brightness alone?

Answer - Computer Vision is a branch of AI that enables systems to interpret and understand images by learning visual patterns (shapes, textures, spatial features) from data, rather than following manually coded rules. A rules-based filter relies on fixed conditions like colour or brightness thresholds set by a human, which only works for very simple, consistent images.


Food images are highly diverse — different lighting, plating, angles, and garnishes — so colour/brightness alone can't reliably distinguish dishes (e.g. biryani and curry can look similar in colour but differ completely in shape/texture). CV-based models (like CNNs) learn these deeper visual patterns automatically, making them far more robust and scalable for categorising diverse, real-world food photos than fixed rule-based logic.



S2 - SCENARIO
You are working on a food quality inspection system that analyses photos of delivered
meals to check for freshness. The CV model was trained on small 128×128 grayscale
images, but restaurant partners upload large RGB photos up to 4000×3000 pixels.

Question: Describe the image preprocessing steps you would apply before passing each
restaurant photo to the model. Why is resizing necessary, and in what situations would
converting an RGB food image to grayscale be beneficial — and when would it lose information
critical to a freshness check?

Answer - 
Preprocessing steps before passing to the model:
Resize the image from 4000×3000 down to 128×128 (using interpolation, e.g. bilinear/bicubic) to match the model's fixed input shape
Convert colour space — RGB to grayscale, since the model expects single-channel input
Normalize pixel values (e.g. scale 0–255 to 0–1) so pixel intensity is on the same scale the model was trained on
Reshape/format the array to match the exact input tensor shape the model expects (e.g. 128×128×1)
Why resizing is necessary: Neural networks require a fixed input size matching their architecture. The model's input layer was built for 128×128, so any other size will either fail to load or force unnecessary computation for resolution the model was never trained to use.
When grayscale helps: When the task depends on shape, texture, or structural patterns rather than colour — e.g., detecting object shape, counting items, or structural defects — grayscale reduces complexity and speeds up training/inference without losing important signal.
When grayscale loses critical information: For freshness checks specifically, colour is often the primary indicator of spoilage — browning, greying, mold spots, discoloration. Converting to grayscale strips this signal, meaning the model could miss key freshness cues that only show up as a colour shift, not a brightness or texture change. In this case, RGB (or at least a colour-based feature) should ideally be preserved rather than converted.

S3 - The correct pipeline: detect orientation → rotate to correct it → crop lower-right quadrant → flip if mirrored → (optional) grayscale/threshold → OCR.


Order justification: The "label is always in the lower-right" assumption is only valid once the bag is in its correct upright orientation. Since bags arrive at random angles, rotation must be corrected first (using cv2.warpAffine with a rotation matrix estimated from the bag's contour/angle) so the image's spatial layout matches what the cropping logic expects. Cropping is a positional operation, so it depends entirely on the image already being correctly oriented. Flipping (if needed for upside-down/mirrored bags) should also happen before or alongside rotation correction, since it affects orientation just like rotation does — both must be resolved before position-based cropping is applied.


What goes wrong if you crop before correcting orientation: The lower-right region of the raw (still-rotated) image doesn't correspond to the label's actual location — it corresponds to whatever happens to be in that quadrant given the random rotation. This could crop out blank bag surface, a partial label, or the wrong area entirely, permanently losing the label content since cropping discards the rest of the image. No amount of rotating afterward can recover it, because the crop already threw away the pixels needed.

S4 - How Canny works:
Gradient calculation — computes the intensity change at each pixel (via Sobel filters) to find potential edges; strong brightness changes indicate likely boundaries.

Non-maximum suppression — thins out thick gradient regions into single-pixel-wide edge lines by keeping only the strongest point in each gradient direction and removing weaker surrounding pixels.
Hysteresis thresholding — classifies edges using two thresholds: pixels above high_threshold are kept as strong edges, pixels below low_threshold are discarded as noise, and pixels in between are kept only if connected to a strong edge, ensuring continuous, non-fragmented edge lines.

Why Gaussian blur before Canny: Raw images contain sensor noise and fine texture that create false, tiny gradient spikes. Since Canny relies entirely on gradient strength, unblurred noise gets misread as edges. Gaussian blur smooths these small variations first so only genuine structural edges (like char boundaries) remain detectable.

Threshold trade-off: Lower thresholds increase sensitivity, catching subtle char marks but also picking up high-contrast false positives like bright plate rims. Higher thresholds reduce false positives (fewer plate rim detections) but risk missing faint or partial char marks that don't produce a strong enough gradient. Since plate rims often have naturally higher contrast than mild charring, tuning purely on thresholds involves a direct trade-off between sensitivity (catching all char) and selectivity (avoiding rim/background edges) — often requiring a masked region of interest in addition to threshold tuning to reliably isolate char marks.

Every frame vs. fixed-interval sampling: Processing every frame guarantees no event is missed but is computationally expensive and storage-heavy, since consecutive frames in mostly-static scenes (like an idle prep station) are highly redundant — this redundancy is what's causing the current 4-second lag and high storage cost. Fixed-interval sampling (e.g., processing every 5th or 10th frame) dramatically reduces load by skipping redundant frames, but risks missing brief, fast-occurring events that fall between sampled frames.

Strategy to avoid missing events: Use adaptive/motion-triggered sampling — sample sparsely during idle periods, but continuously run lightweight motion detection (e.g., frame differencing with cv2.absdiff or background subtraction with cv2.

createBackgroundSubtractorMOG2) between samples. When meaningful motion is detected, temporarily switch to full-frame processing to capture the event in detail, then return to sparse sampling once activity settles. This balances efficiency with event coverage.
Role of cap.set(cv2.CAP_PROP_POS_FRAMES, n): It allows the video capture pointer to jump directly to a specific frame index rather than reading and discarding frames sequentially via repeated cap.read() calls. This makes fixed-interval frame skipping efficient to implement in code — you can directly seek to frame n, n+10, n+20, etc., avoiding unnecessary decode overhead for skipped frames (most reliable with recorded video files rather than live streams).